In [1]:
!pip install -q uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 22.0 MB/s eta 0:00:00


In [2]:
!uv init goodreads-rag

Initialized project `goodreads-rag` at `/content/goodreads-rag`


In [3]:
%cd goodreads-rag

/content/goodreads-rag


In [ ]:
import pandas as pd
# import csv

# Increase the field size limit to handle potentially very large fields
# csv.field_size_limit(131072 * 10) # Increasing by a factor of 10

df = pd.read_csv("data/cleaned_goodreads_books_dataset.csv")

print(df.head())

In [5]:
# load the document into a langchain document
import pandas as pd
from langchain_core.documents import Document
df = pd.read_csv('data/cleaned_goodreads_books_dataset.csv')
documents = []
for _,row in df.iterrows():
    text = f"""
Title: {row['title']}
Author: {row['author']}
Genre: {row['genres']}
Rating: {row['rating']}
Description: {row['description']}
Popular Reviews: {row['popularReviews']}
"""
    documents.append(
        Document(
            page_content=text,
            metadata = {
                "title": row['title'],
                "author": row['author'],
                "rating": row["rating"]
            }
        )
    )

print(len(documents))

724


In [ ]:
import pandas as pd
import csv

# Increase the field size limit to handle potentially very large fields
csv.field_size_limit(131072 * 10) # Increasing by a factor of 10


!uv pip install \
pandas \
langchain \
langchain-community \
langchain-huggingface \
sentence-transformers \
faiss-cpu \
transformers \
accelerate \
torch \
ollama \
tqdm \
pydantic \
bitsandbytes

In [7]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import pipeline
import torch

# -------------------------
# Load Embedding Model
# -------------------------
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
db = FAISS.from_documents(documents, embedding)

# Save the FAISS index to the local directory
db.save_local("vector_db")


# -------------------------
# Load TinyLlama
# -------------------------
generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device=0 if torch.cuda.is_available() else -1,
)

# -------------------------
# Query
# -------------------------
query = "Recommend some magic books"

docs = db.similarity_search(query, k=5)

# Limit context length
context = docs[0].page_content[:500]

prompt = f"""
You are a helpful book recommendation assistant.

Use ONLY the context below.

Context:
{context}

Question:
{query}

Answer:
"""

# -------------------------
# Generate
# -------------------------
result = generator(
    prompt,
    max_new_tokens=200,
    do_sample=False,
    truncation=True,
)

print(result[0]["generated_text"])

/tmp/ipykernel_594/1381508073.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (13 outputs left out)

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



You are a helpful book recommendation assistant.

Use ONLY the context below.

Context:

Title: The Book of Witches
Author: {'name': 'Jonathan Strahan', 'about': '', 'aboutHTML': '', 'imageURL': 'https://i.gr-assets.com/images/S/compressed.photo.goodreads.com/nophoto/user/u_700x933.png', 'followers': 462}
Genre: ['Fantasy', 'Short Stories', 'Witches', 'Anthologies', 'Fiction', 'Horror', 'Adult', 'Paranormal', 'Poetry', 'Halloween']
Rating: 3.46
Description: With a breathtaking array of original stories from around the world, P. Djèlí Clark, Amal El Mohtar, Garth Nix, Darcie Little 

Question:
Recommend some magic books

Answer:
1. The Book of Witches by Jonathan Strahan
2. The Witch's Heart by Amal El Mohtar
3. The Witch's House by Garth Nix
4. The Witch's House by Darcie Little
5. The Witch's House by Amal El Mohtar
6. The Witch's House by Garth Nix
7. The Witch's House by Darcie Little
8. The Witch's House by Amal El Mohtar
9. The Witch's House by Garth Nix
10. The Witch's House by 

In [9]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import pipeline
import torch

# -------------------------
# Load Embedding Model
# -------------------------
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}  # Keep embeddings on CPU
)

# -------------------------
# Load FAISS DB
# -------------------------
db = FAISS.load_local(
    "vector_db",
    embedding,
    allow_dangerous_deserialization=True
)

# -------------------------
# Load TinyLlama
# -------------------------
generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device=0 if torch.cuda.is_available() else -1,
)

print("=" * 60)
print("📚 Goodreads RAG Chatbot")
print("Type 'exit' to quit.")
print("=" * 60)

while True:

    question = input("\nQuestion: ").strip()

    if question.lower() == "exit":
        print("Goodbye!")
        break

    # Retrieve only 2 documents
    docs = db.similarity_search(question, k=2)

    # Keep only the first 300 characters of each document
    context = "\n\n".join(
        doc.page_content[:300]
        for doc in docs
    )

    prompt = f"""
You are a helpful Goodreads book recommendation assistant.

Answer ONLY using the books provided below.

If the answer is not available in the context, reply:
"I couldn't find a suitable recommendation."

Context:
{context}

Question:
{question}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=200,
        do_sample=False,
        return_full_text=False,
    )

    print("\nAnswer:\n")
    print(response[0]["generated_text"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

📚 Goodreads RAG Chatbot
Type 'exit' to quit.

Question: suggest thriller books


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:

1. The Silent Patient by Alex Michaelides
2. The Woman in the Window by A.J. Finn
3. The Girl on the Train by Paula Hawkins
4. The Wife Between Us by Greer Hendricks and Sarah Pekkanen
5. The Girl on the Train by Paula Hawkins

Context:

Title: The Girl on the Train
Author: {'name': 'Paula Hawkins', 'about': 'Paula Hawkins is an English author. She is best known for her novel The Girl on the Train, which was adapted into a film in 2016. The book was published in 2015 and has since been translated into over 40 languages. The Girl on the Train is a psychological thriller that follows Rachel, a woman who lives alone in a small town and becomes obsessed with a woman she sees every day, leading her down a dangerous path

Question: exit
Goodbye!
